In [1]:
from pathlib import Path

BASE = Path("/content/GC-MS_compounds_first_alignment")
SRC  = BASE / "src"
PKG  = SRC / "gcms_cf"

DATA_RAW_ZIP   = BASE / "data" / "raw" / "incoming_zip"
DATA_EXTRACTED = BASE / "data" / "raw" / "extracted"
DATA_OUTPUT    = BASE / "data" / "output"
CACHE_DIR      = BASE / ".cache_pipeline" / "ui_cache"

# Crea cartelle necessarie (senza toccare logica)
for d in [DATA_RAW_ZIP, DATA_EXTRACTED, DATA_OUTPUT, CACHE_DIR, PKG]:
    d.mkdir(parents=True, exist_ok=True)

# Rende "gcms_cf" importabile in modo robusto
init_file = PKG / "__init__.py"
if not init_file.exists():
    init_file.write_text("# package init (auto)\n", encoding="utf-8")

print("✅ Struttura cartelle pronta:")
print("BASE:", BASE)
print("SRC :", SRC)
print("PKG :", PKG)


✅ Struttura cartelle pronta:
BASE: /content/GC-MS_compounds_first_alignment
SRC : /content/GC-MS_compounds_first_alignment/src
PKG : /content/GC-MS_compounds_first_alignment/src/gcms_cf


In [2]:
import base64, io, zipfile, textwrap
from pathlib import Path

BASE = Path("/content/GC-MS_compounds_first_alignment")
PKG  = BASE / "src" / "gcms_cf"

if not PKG.exists():
    raise FileNotFoundError(f"Non trovo {PKG}. Prima ricostruisci i moduli (Blocco 2) o assicurati che esistano.")

# crea zip in memoria dei soli .py sotto src/gcms_cf
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(PKG.rglob("*.py")):
        arcname = str(p.relative_to(BASE))  # es: src/gcms_cf/alignment.py
        z.write(p, arcname=arcname)

payload = base64.b64encode(buf.getvalue()).decode("ascii")
wrapped = "\n".join(textwrap.wrap(payload, 100))

print("\n" + "="*80)
print("COPIA/INCOLLA QUESTO COME 'BLOCCO 4 — RESTORE MODULI gcms_cf'\n")
print("="*80 + "\n")

print("import base64, io, zipfile")
print("from pathlib import Path")
print("")
print("BASE = Path('/content/GC-MS_compounds_first_alignment')")
print("BASE.mkdir(parents=True, exist_ok=True)")
print("(BASE/'src'/'gcms_cf').mkdir(parents=True, exist_ok=True)")
print("")
print("PAYLOAD_B64 = '''")
print(wrapped)
print("'''")
print("")
print("data = base64.b64decode(PAYLOAD_B64.replace('\\n','').strip())")
print("with zipfile.ZipFile(io.BytesIO(data), 'r') as z:")
print("    z.extractall(BASE)")
print("")
print("print('✅ Ripristinati moduli in:', BASE/'src'/'gcms_cf')")



COPIA/INCOLLA QUESTO COME 'BLOCCO 4 — RESTORE MODULI gcms_cf'


import base64, io, zipfile
from pathlib import Path

BASE = Path('/content/GC-MS_compounds_first_alignment')
BASE.mkdir(parents=True, exist_ok=True)
(BASE/'src'/'gcms_cf').mkdir(parents=True, exist_ok=True)

PAYLOAD_B64 = '''
UEsDBBQAAAAIAM06LlyVfYEAGAAAABYAAAAXAAAAc3JjL2djbXNfY2YvX19pbml0X18ucHlTVihITM5OTE9VyMzLLFHQSCwtydfk
AgBQSwECFAMUAAAACADNOi5clX2BABgAAAAWAAAAFwAAAAAAAAAAAAAApIEAAAAAc3JjL2djbXNfY2YvX19pbml0X18ucHlQSwUG
AAAAAAEAAQBFAAAATQAAAAAA
'''

data = base64.b64decode(PAYLOAD_B64.replace('\n','').strip())
with zipfile.ZipFile(io.BytesIO(data), 'r') as z:
    z.extractall(BASE)

print('✅ Ripristinati moduli in:', BASE/'src'/'gcms_cf')


In [1]:
import json, re
from pathlib import Path
from google.colab import files

BASE = Path("/content/GC-MS_compounds_first_alignment")
PKG  = BASE / "src" / "gcms_cf"
PKG.mkdir(parents=True, exist_ok=True)
(PKG / "__init__.py").write_text("# package init\n", encoding="utf-8")

print("Carica ORA il notebook sorgente: GC_MS_alignment_1.ipynb (NON lo zip dati)")
up = files.upload()

if not up:
    raise RuntimeError("Nessun file caricato.")

name = list(up.keys())[0]
if not name.lower().endswith(".ipynb"):
    raise ValueError("Qui serve un file .ipynb. Lo ZIP dei dati va caricato nel wizard (Blocco 3).")

nb = json.loads(up[name].decode("utf-8"))
cells = nb.get("cells", [])

write_re = re.compile(r"^%%writefile\s+([^\r\n]+)", re.M)

last = {}
for c in cells:
    if c.get("cell_type") != "code":
        continue
    src = "".join(c.get("source", []))
    m = write_re.search(src)
    if not m:
        continue
    wf_path = m.group(1).strip().replace("\\", "/")
    if "/src/gcms_cf/" not in wf_path:
        continue
    rel = wf_path.split("/src/gcms_cf/", 1)[1]
    if rel.endswith(".py"):
        last[rel] = src

print("Moduli trovati nel notebook:", len(last))

for rel, src in last.items():
    content = "".join(src.splitlines(True)[1:])  # salta %%writefile
    (PKG / rel).write_text(content, encoding="utf-8")

py = sorted([p.name for p in PKG.glob("*.py")])
print("✅ Ora i file in gcms_cf sono:", len(py))
print("Elenco (prime 30):", py[:30])


Carica ORA il notebook sorgente: GC_MS_alignment_1.ipynb (NON lo zip dati)


Saving GC_MS_alignment_1.ipynb to GC_MS_alignment_1.ipynb
Moduli trovati nel notebook: 20
✅ Ora i file in gcms_cf sono: 21
Elenco (prime 30): ['__init__.py', 'alignment.py', 'cache.py', 'coelution.py', 'config.py', 'core_compounds.py', 'deconvolution.py', 'eic.py', 'export_xlsx.py', 'hr_deconv_extract.py', 'hr_eic.py', 'hr_spectrum.py', 'io_readers.py', 'models.py', 'mzdata_io.py', 'peak_picking.py', 'pipeline.py', 'rescue.py', 'scan_io.py', 'scan_models.py', 'tic_peaks.py']


In [5]:
# BLOCCO 4 — RESTORE MODULI gcms_cf
PAYLOAD_B64 = '''...'''


In [6]:
# =========================
# GC-MS TOF HR Wizard (mzData.xml in ZIP) — UI completa Step1 + Step2
# =========================

import os, sys, re, json, hashlib, pickle, shutil, zipfile, subprocess
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from tqdm.auto import tqdm
import importlib

BASE = Path("/content/GC-MS_compounds_first_alignment")
SRC  = BASE / "src"

DATA_RAW_ZIP   = BASE / "data" / "raw" / "incoming_zip"
DATA_EXTRACTED = BASE / "data" / "raw" / "extracted"
DATA_OUTPUT    = BASE / "data" / "output"
CACHE_DIR      = BASE / ".cache_pipeline" / "ui_cache"

for d in [DATA_RAW_ZIP, DATA_EXTRACTED, DATA_OUTPUT, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

STATE = {
    "xml_files": [],
    "selected_files": [],
    "all_compounds": [],
    "compounds_by_sample": {},
    "summaries_by_sample": {},
    "last_excel": None,
}

# -------------------------
# Helpers
# -------------------------

def ensure_imports(out):
    with out:
        print("📦 Controllo import del progetto...")

    candidates = [SRC, Path("/content/src")]

    try:
        for p in Path("/content").rglob("gcms_cf"):
            if p.is_dir():
                candidates.append(p.parent)
    except Exception:
        pass

    for c in candidates:
        c = str(c)
        if Path(c).exists() and c not in sys.path:
            sys.path.insert(0, c)

    importlib.invalidate_caches()

    try:
        import gcms_cf  # noqa
        with out:
            print("✅ Import gcms_cf OK.")
        return True
    except Exception as e:
        with out:
            print("❌ Import fallito:", repr(e))
            print("➡️ Cartelle candidate controllate (sys.path):")
            for c in candidates[:8]:
                print(" -", c)
            print("➡️ Se non esiste /content/GC-MS_compounds_first_alignment/src/gcms_cf/, riesegui la CELLA 2.")
        return False

def params_hash(d: dict) -> str:
    payload = json.dumps(d, sort_keys=True, ensure_ascii=False).encode("utf-8")
    return hashlib.sha1(payload).hexdigest()[:12]

def guess_sample_id(p: Path) -> str:
    name = p.name
    m = re.search(r"(AU\d+)", name)
    if m:
        return m.group(1)
    stem = p.stem.replace(".mzdata", "")
    return stem

def refresh_file_list():
    xmls_all = sorted(DATA_EXTRACTED.rglob("*.xml"))
    mzdata_xml = [p for p in xmls_all if p.name.lower().endswith("mzdata.xml")]
    mzdata_plain = sorted(DATA_EXTRACTED.rglob("*.mzdata"))

    files_found = sorted(set(mzdata_xml + mzdata_plain), key=lambda p: str(p))

    STATE["xml_files"] = files_found
    file_select.options = [(p.name, str(p)) for p in files_found]
    btn_run_extract.disabled = (len(files_found) == 0)

def get_selected_files():
    chosen = list(file_select.value)
    if chosen:
        return [Path(p) for p in chosen]
    return STATE["xml_files"][:]

# -------------------------
# UI layout helpers
# -------------------------
def make_param_row(slider, text_box, help_html):
    slider.layout = widgets.Layout(flex="3 1 auto", width="auto")
    text_box.layout = widgets.Layout(width="120px")
    help_html.layout = widgets.Layout(flex="4 1 auto", width="auto")
    row = widgets.HBox(
        [slider, text_box, help_html],
        layout=widgets.Layout(width="100%", justify_content="space-between", align_items="center")
    )
    widgets.link((slider, "value"), (text_box, "value"))
    return row

def set_button_full(btn, minw=420):
    btn.layout = widgets.Layout(width="100%", min_width=f"{minw}px")
    btn.style.button_width = "auto"

# -------------------------
# Widgets
# -------------------------
title = widgets.HTML(
    "<h3>GC-MS TOF HR Wizard (mzData.xml dentro ZIP) — Colab</h3>"
    "<div style='padding:8px 10px; background:#fff3cd; border:1px solid #ffeeba; border-radius:8px;'>"
    "<b>Come si avvia:</b> premi una volta il tasto ▶️ (Run) a sinistra di questa cella per far comparire l’interfaccia. "
    "Poi usa i bottoni <b>RUN Step 1</b> e <b>RUN Step 2</b> qui sotto."
    "</div>"
)

out = widgets.Output(layout={
    "border": "1px solid #ddd",
    "padding": "10px",
    "max_height": "420px",
    "overflow_y": "auto"
})

lbl_step1 = widgets.HTML(
    "<span style='display:inline-block; padding:6px 10px; background:#1a73e8; color:white; border-radius:6px; font-weight:600;'>"
    "1) Carica ZIP</span>"
)

btn_upload_zip = widgets.Button(description="Scegli file", icon="upload", button_style="primary")
btn_upload_zip.layout = widgets.Layout(width="220px")

chk_clean_extract = widgets.Checkbox(value=True, description="Pulisci 'extracted' prima dell’unzip")
btn_refresh_files = widgets.Button(description="Aggiorna lista file", icon="refresh")
btn_refresh_files.layout = widgets.Layout(width="220px")

lbl_zip = widgets.HTML("<b>ZIP:</b> nessuno")

file_select = widgets.SelectMultiple(
    options=[],
    description="File mzData",
    rows=7,
    layout=widgets.Layout(width="100%")
)

btn_run_extract = widgets.Button(description="RUN Step 1 — Estrai compound", button_style="success", icon="play")
btn_run_align   = widgets.Button(description="RUN Step 2 — Allinea + Excel", button_style="warning", icon="cogs")
btn_download    = widgets.Button(description="Scarica Excel", icon="download")

for b in [btn_run_extract, btn_run_align, btn_download]:
    set_button_full(b, minw=420)

btn_run_align.disabled = True
btn_download.disabled = True
btn_run_extract.disabled = True

# -------------------------
# Step 1 parameters
# -------------------------
s_top_k_windows = widgets.IntSlider(value=250, min=20, max=400, step=10, description="TIC windows", continuous_update=False)
t_top_k_windows = widgets.IntText(value=250)
h_top_k_windows = widgets.HTML("<small>Finestre TIC analizzate (↑ = più compound, ↑ tempo).</small>")

s_half_width = widgets.FloatSlider(value=0.35, min=0.10, max=0.80, step=0.05, description="half_width", continuous_update=False)
t_half_width = widgets.FloatText(value=0.35)
h_half_width = widgets.HTML("<small>Finestra RT: apice ± half_width (min). Influenza quanti scans entrano.</small>")

s_ppm_tol = widgets.FloatSlider(value=10.0, min=3.0, max=20.0, step=1.0, description="ppm_tol", continuous_update=False)
t_ppm_tol = widgets.FloatText(value=10.0)
h_ppm_tol = widgets.HTML("<small>Tolleranza ±ppm per EIC e match nello scan (↑ = più inclusivo).</small>")

s_top_seeds = widgets.IntSlider(value=300, min=60, max=600, step=20, description="top_seeds", continuous_update=False)
t_top_seeds = widgets.IntText(value=300)
h_top_seeds = widgets.HTML("<small>Seed m/z dallo scan apice (↑ = più ioni candidati, ↑ tempo).</small>")

s_hit_rt_tol = widgets.FloatSlider(value=0.08, min=0.03, max=0.15, step=0.01, description="hit_rt_tol", continuous_update=False)
t_hit_rt_tol = widgets.FloatText(value=0.08)
h_hit_rt_tol = widgets.HTML("<small>Accetta hit se apex EIC è entro ±hit_rt_tol dall’apice TIC.</small>")

s_hit_rel = widgets.FloatSlider(value=0.04, min=0.01, max=0.15, step=0.01, description="hit_rel_h", continuous_update=False)
t_hit_rel = widgets.FloatText(value=0.04)
h_hit_rel = widgets.HTML("<small>Soglia peak-picking EIC (↓ = più hits, anche più rumore).</small>")

s_max_hits = widgets.IntSlider(value=150, min=30, max=300, step=10, description="max_hits", continuous_update=False)
t_max_hits = widgets.IntText(value=150)
h_max_hits = widgets.HTML("<small>Max hit per finestra (↑ = più cluster possibili, ↑ tempo).</small>")

s_min_corr = widgets.FloatSlider(value=0.78, min=0.60, max=0.95, step=0.01, description="min_corr", continuous_update=False)
t_min_corr = widgets.FloatText(value=0.78)
h_min_corr = widgets.HTML("<small>Correlazione EIC per cluster (↓ = più co-eluizioni accettate).</small>")

s_max_comp_win = widgets.IntSlider(value=5, min=1, max=10, step=1, description="cmp/window", continuous_update=False)
t_max_comp_win = widgets.IntText(value=5)
h_max_comp_win = widgets.HTML("<small>Max compound estratti per finestra TIC (↑ = più risultati).</small>")

s_min_ions = widgets.IntSlider(value=5, min=3, max=12, step=1, description="min_ions", continuous_update=False)
t_min_ions = widgets.IntText(value=5)
h_min_ions = widgets.HTML("<small>Min ioni nel cluster per accettare un compound (↑ = più selettivo).</small>")

s_min_area_frac = widgets.FloatSlider(value=0.05, min=0.01, max=0.50, step=0.01, description="area_frac", continuous_update=False)
t_min_area_frac = widgets.FloatText(value=0.05)
h_min_area_frac = widgets.HTML("<small>Cluster secondari: area ≥ area_frac × area_best (↓ = più composti).</small>")

s_min_scan_pur = widgets.FloatSlider(value=0.02, min=0.00, max=0.30, step=0.01, description="scan_pur", continuous_update=False)
t_min_scan_pur = widgets.FloatText(value=0.02)
h_min_scan_pur = widgets.HTML("<small>Min scan_purity=matched_sum/TICscan (↓ = più compound “sporchi”).</small>")

s_min_matched = widgets.IntSlider(value=4, min=1, max=20, step=1, description="min_match", continuous_update=False)
t_min_matched = widgets.IntText(value=4)
h_min_matched = widgets.HTML("<small>Min picchi matchati nello scan per accettare il compound.</small>")

step1_rows = widgets.VBox([
    widgets.HTML("<b>Step 1 — Estrazione compound (HR, seed-EIC)</b><br><small>Slider + valore numerico + descrizione.</small>"),
    make_param_row(s_top_k_windows, t_top_k_windows, h_top_k_windows),
    make_param_row(s_half_width,    t_half_width,    h_half_width),
    make_param_row(s_ppm_tol,       t_ppm_tol,       h_ppm_tol),
    make_param_row(s_top_seeds,     t_top_seeds,     h_top_seeds),
    make_param_row(s_hit_rt_tol,    t_hit_rt_tol,    h_hit_rt_tol),
    make_param_row(s_hit_rel,       t_hit_rel,       h_hit_rel),
    make_param_row(s_max_hits,      t_max_hits,      h_max_hits),
    make_param_row(s_min_corr,      t_min_corr,      h_min_corr),
    make_param_row(s_max_comp_win,  t_max_comp_win,  h_max_comp_win),
    make_param_row(s_min_ions,      t_min_ions,      h_min_ions),
    make_param_row(s_min_area_frac, t_min_area_frac, h_min_area_frac),
    make_param_row(s_min_scan_pur,  t_min_scan_pur,  h_min_scan_pur),
    make_param_row(s_min_matched,   t_min_matched,   h_min_matched),
])

# -------------------------
# Step 2 parameters
# -------------------------
s_mz_ppm      = widgets.FloatSlider(value=10.0, min=3.0, max=20.0, step=1.0, description="mz_ppm", continuous_update=False)
t_mz_ppm      = widgets.FloatText(value=10.0)
h_mz_ppm      = widgets.HTML("<small>Matching m/z in ppm per cosine similarity/allineamento (↑ = più match, ↑ rischio).</small>")

s_rt_strict   = widgets.FloatSlider(value=0.35, min=0.05, max=1.00, step=0.05, description="RT strict", continuous_update=False)
t_rt_strict   = widgets.FloatText(value=0.35)
h_rt_strict   = widgets.HTML("<small>Tolleranza RT strict (min) per match tra campioni (↑ = più match).</small>")

s_rt_relax    = widgets.FloatSlider(value=0.55, min=0.10, max=1.50, step=0.05, description="RT relax", continuous_update=False)
t_rt_relax    = widgets.FloatText(value=0.55)
h_rt_relax    = widgets.HTML("<small>Tolleranza RT usata nella rescue (di solito ≥ strict).</small>")

s_cos_strict  = widgets.FloatSlider(value=0.75, min=0.50, max=0.95, step=0.01, description="cos strict", continuous_update=False)
t_cos_strict  = widgets.FloatText(value=0.75)
h_cos_strict  = widgets.HTML("<small>Cosine minima strict (↓ = più match, ↑ rischio di falsi).</small>")

s_cos_relax   = widgets.FloatSlider(value=0.70, min=0.40, max=0.95, step=0.01, description="cos relax", continuous_update=False)
t_cos_relax   = widgets.FloatText(value=0.70)
h_cos_relax   = widgets.HTML("<small>Cosine minima per rescue (di solito ≤ strict).</small>")

s_dlog_strict = widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, description="dlog strict", continuous_update=False)
t_dlog_strict = widgets.FloatText(value=1.0)
h_dlog_strict = widgets.HTML("<small>Δlog10(area) max strict: 1.0≈10× (↑ = più match).</small>")

s_dlog_relax  = widgets.FloatSlider(value=2.0, min=0.2, max=4.0, step=0.1, description="dlog relax", continuous_update=False)
t_dlog_relax  = widgets.FloatText(value=2.0)
h_dlog_relax  = widgets.HTML("<small>Δlog10(area) max rescue: 2.0≈100× (↑ = più rescue).</small>")

s_core_frac   = widgets.FloatSlider(value=0.67, min=0.34, max=1.0, step=0.01, description="core_frac", continuous_update=False)
t_core_frac   = widgets.FloatText(value=0.67)
h_core_frac   = widgets.HTML("<small>Core: con 3 campioni, 0.67 ≈ presenti in ≥2/3.</small>")

t_max_score = widgets.Text(value="1.2", description="max_score")
t_margin    = widgets.Text(value="0.15", description="min_margin")
h_freni     = widgets.HTML("<small>Freni rescue opzionali: svuota per disattivarli (più permissivo, più rischio).</small>")

step2_rows = widgets.VBox([
    widgets.HTML("<b>Step 2 — Allineamento + Core/Rescue + Excel</b><br><small>Slider + valore numerico + descrizione.</small>"),
    make_param_row(s_mz_ppm,      t_mz_ppm,      h_mz_ppm),
    make_param_row(s_rt_strict,   t_rt_strict,   h_rt_strict),
    make_param_row(s_rt_relax,    t_rt_relax,    h_rt_relax),
    make_param_row(s_cos_strict,  t_cos_strict,  h_cos_strict),
    make_param_row(s_cos_relax,   t_cos_relax,   h_cos_relax),
    make_param_row(s_dlog_strict, t_dlog_strict, h_dlog_strict),
    make_param_row(s_dlog_relax,  t_dlog_relax,  h_dlog_relax),
    make_param_row(s_core_frac,   t_core_frac,   h_core_frac),
    widgets.HTML("<small><i>Freni rescue (opzionali):</i></small>"),
    widgets.HBox([t_max_score, t_margin, h_freni], layout=widgets.Layout(width="100%", align_items="center"))
])

accordion = widgets.Accordion(children=[step1_rows, step2_rows])
accordion.set_title(0, "Parametri Step 1 (Estrazione)")
accordion.set_title(1, "Parametri Step 2 (Allineamento)")
accordion.selected_index = 0

# -------------------------
# Actions
# -------------------------
def upload_zip_and_extract(_):
    with out:
        clear_output()
        print("📤 Seleziona lo ZIP (upload Colab standard)...")

    up = files.upload()
    if not up:
        with out:
            print("❌ Nessun file selezionato.")
        return

    zip_name = list(up.keys())[0]
    content = up[zip_name]

    dst = DATA_RAW_ZIP / zip_name
    with open(dst, "wb") as f:
        f.write(content)

    lbl_zip.value = f"<b>ZIP:</b> {zip_name} ({dst.stat().st_size/1e6:.1f} MB)"

    if chk_clean_extract.value:
        shutil.rmtree(DATA_EXTRACTED, ignore_errors=True)
        DATA_EXTRACTED.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(dst, "r") as z:
            z.extractall(DATA_EXTRACTED)
    except Exception as e:
        with out:
            print("❌ Unzip fallito:", repr(e))
        return

    refresh_file_list()
    with out:
        print(f"✅ Unzip completato. File mzData trovati: {len(STATE['xml_files'])}")
        if STATE["xml_files"]:
            print("Seleziona i file (o lascia tutti) e premi: RUN Step 1 — Estrai compound")

def run_extract(_):
    if not ensure_imports(out):
        return

    selected = get_selected_files()
    if not selected:
        with out:
            print("❌ Nessun file mzData trovato. Carica lo ZIP.")
        return

    step1_params = dict(
        top_k_windows=int(s_top_k_windows.value),
        half_width_min=float(s_half_width.value),
        ppm_tol=float(s_ppm_tol.value),
        top_n_seeds=int(s_top_seeds.value),
        hit_rt_tol=float(s_hit_rt_tol.value),
        hit_min_rel_height=float(s_hit_rel.value),
        max_hits=int(s_max_hits.value),
        min_corr=float(s_min_corr.value),
        max_compounds_per_window=int(s_max_comp_win.value),
        min_cluster_ions=int(s_min_ions.value),
        min_cluster_area_frac=float(s_min_area_frac.value),
        min_scan_purity=float(s_min_scan_pur.value),
        min_matched_peaks=int(s_min_matched.value),
    )
    ph = params_hash(step1_params)

    from gcms_cf.hr_deconv_extract import extract_compounds_from_mzdata

    compounds_by_sample = {}
    summaries_by_sample = {}

    with out:
        clear_output()
        print("🚀 STEP 1 — Estrazione compound")
        print("File:", [p.name for p in selected])
        print("Parametri:", step1_params)
        print("Cache key:", ph)
        print("—")

    for xml_path in tqdm(selected, desc="Estrazione per file"):
        sample_id = guess_sample_id(xml_path)
        pkl_c = CACHE_DIR / f"{sample_id}_compounds_{ph}.pkl"
        pkl_s = CACHE_DIR / f"{sample_id}_summary_{ph}.pkl"

        if pkl_c.exists() and pkl_s.exists():
            with open(pkl_c, "rb") as f:
                comps = pickle.load(f)
            with open(pkl_s, "rb") as f:
                summ = pickle.load(f)
            with out:
                print(f"[CACHE] {sample_id}: compounds={len(comps)}")
        else:
            comps, summ = extract_compounds_from_mzdata(str(xml_path), sample_id=sample_id, **step1_params)
            with open(pkl_c, "wb") as f:
                pickle.dump(comps, f)
            with open(pkl_s, "wb") as f:
                pickle.dump(summ, f)
            with out:
                print(f"[RUN]   {sample_id}: compounds={len(comps)}")

        compounds_by_sample[sample_id] = comps
        summaries_by_sample[sample_id] = summ

    STATE["compounds_by_sample"] = compounds_by_sample
    STATE["summaries_by_sample"] = summaries_by_sample
    STATE["all_compounds"] = [c for sid in compounds_by_sample for c in compounds_by_sample[sid]]

    with out:
        print("\n✅ STEP 1 completato.")
        print("Totale compounds:", len(STATE["all_compounds"]))
        print("➡️ Ora passa allo Step 2 (accordion) e premi: RUN Step 2 — Allinea + Excel")
    btn_run_align.disabled = False
    accordion.selected_index = 1

def to_float_or_none(s: str):
    try:
        ss = str(s).strip()
        if ss == "":
            return None
        return float(ss)
    except Exception:
        return None

def run_align(_):
    if not ensure_imports(out):
        return

    all_compounds = STATE.get("all_compounds", [])
    if not all_compounds:
        with out:
            print("❌ Nessun compound in memoria. Esegui prima lo Step 1.")
        return

    step2_params = dict(
        mz_ppm=float(s_mz_ppm.value),
        rt_tol_strict=float(s_rt_strict.value),
        rt_tol_relax=float(s_rt_relax.value),
        min_cosine_strict=float(s_cos_strict.value),
        min_cosine_relax=float(s_cos_relax.value),
        max_dlog10_area_strict=float(s_dlog_strict.value),
        max_dlog10_area_relax=float(s_dlog_relax.value),
        core_frac=float(s_core_frac.value),
        max_rescue_score=to_float_or_none(t_max_score.value),
        min_score_margin=to_float_or_none(t_margin.value),
    )

    from gcms_cf.alignment import align_compounds_clusters, clusters_to_table
    from gcms_cf.core_compounds import add_core_flags_df
    from gcms_cf.rescue import rescue_core_missing
    from gcms_cf.export_xlsx import export_feature_table

    compounds_by_sample = {}
    for c in all_compounds:
        compounds_by_sample.setdefault(c.sample_id, []).append(c)
    sample_ids = sorted(compounds_by_sample.keys())

    with out:
        clear_output()
        print("🧩 STEP 2 — Allineamento + Core/Rescue + Excel")
        print("Sample IDs:", sample_ids)
        print("Parametri:", step2_params)
        print("—")

    clusters = align_compounds_clusters(
        all_compounds,
        rt_tol=step2_params["rt_tol_strict"],
        use_ri=False,
        ri_tol=20.0,
        area_agg="max",
        min_cosine=step2_params["min_cosine_strict"],
        mz_tol=0.01,
        mz_ppm=step2_params["mz_ppm"],
        max_dlog10_area=step2_params["max_dlog10_area_strict"],
    )

    df_strict = clusters_to_table(clusters, fill_missing=0.0)
    df_strict2, sample_cols = add_core_flags_df(df_strict, core_frac=step2_params["core_frac"])
    core_ids = list(df_strict2[df_strict2["is_core"] == True]["feature_id"].astype(str))

    with out:
        print(f"Feature totali (strict): {len(df_strict)}")
        print(f"Core (>= {step2_params['core_frac']:.2f}): {len(core_ids)}")

    rescued_map = rescue_core_missing(
        clusters,
        compounds_by_sample,
        core_feature_ids=core_ids,
        sample_ids=sample_ids,
        area_agg="max",
        rt_tol=step2_params["rt_tol_relax"],
        use_ri=False,
        ri_tol=20.0,
        min_cosine=step2_params["min_cosine_relax"],
        # >>> UNICA MODIFICA RICHIESTA: aggiunto mz_tol <<<
        mz_tol=0.01,
        mz_ppm=step2_params["mz_ppm"],
        max_dlog10_area=step2_params["max_dlog10_area_relax"],
        max_rescue_score=step2_params["max_rescue_score"],
        min_score_margin=step2_params["min_score_margin"],
    )

    df_final = clusters_to_table(clusters, fill_missing=0.0)
    df_final = df_final.merge(
        df_strict2[["feature_id", "n_present", "presence_frac", "is_core"]],
        on="feature_id",
        how="left",
    ).rename(columns={"n_present":"n_present_strict", "presence_frac":"presence_frac_strict"})

    present_final = (df_final[sample_cols] > 0.0)
    df_final["n_present_final"] = present_final.sum(axis=1)
    df_final["presence_frac_final"] = df_final["n_present_final"] / float(len(sample_cols))

    df_final["n_rescued"] = df_final["feature_id"].map(lambda fid: len(rescued_map.get(fid, []))).fillna(0).astype(int)
    df_final["rescued_samples"] = df_final["feature_id"].map(lambda fid: ";".join(rescued_map.get(fid, [])) if fid in rescued_map else "")

    df_final = df_final.sort_values(
        ["is_core","presence_frac_final","rt_ref"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    out_xlsx = DATA_OUTPUT / "FEATURE_TABLE_UI.xlsx"
    export_feature_table(df_final, str(out_xlsx), sheet_name="FeatureTable")
    STATE["last_excel"] = str(out_xlsx)

    n_resc_feat = sum(1 for v in rescued_map.values() if v)
    with out:
        print(f"Feature con almeno 1 rescue: {n_resc_feat}")
        print(f"✅ Excel creato: {out_xlsx}")
        display(df_final.head(15))
        print("➡️ Ora premi: Scarica Excel")

    btn_download.disabled = False

def download_excel(_):
    p = STATE.get("last_excel")
    if not p or not os.path.exists(p):
        with out:
            print("❌ Nessun Excel disponibile. Esegui prima lo Step 2.")
        return
    files.download(p)

def refresh_files_btn(_):
    refresh_file_list()
    with out:
        print(f"🔎 File aggiornati: {len(STATE['xml_files'])} mzData trovati.")
    btn_run_extract.disabled = (len(STATE["xml_files"]) == 0)

# Bind callbacks
btn_upload_zip.on_click(upload_zip_and_extract)
btn_run_extract.on_click(run_extract)
btn_run_align.on_click(run_align)
btn_download.on_click(download_excel)
btn_refresh_files.on_click(refresh_files_btn)

# init
refresh_file_list()

# Layout
top_row = widgets.HBox(
    [lbl_step1, btn_upload_zip, chk_clean_extract, btn_refresh_files],
    layout=widgets.Layout(width="100%", justify_content="flex-start", align_items="center", column_gap="14px")
)

left_col = widgets.VBox([lbl_zip, file_select], layout=widgets.Layout(width="62%"))
right_col = widgets.VBox([btn_run_extract, btn_run_align, btn_download], layout=widgets.Layout(width="38%"))
mid_row = widgets.HBox([left_col, right_col], layout=widgets.Layout(width="100%", column_gap="16px"))

# evita UI duplicate se rilanci la cella
clear_output(wait=True)

display(title, top_row, mid_row, accordion, out)

with out:
    print("👉 Flusso consigliato:")
    print("1) Premi 'Scegli file' e seleziona lo ZIP.")
    print("2) Premi 'RUN Step 1 — Estrai compound'.")
    print("3) Premi 'RUN Step 2 — Allinea + Excel' (si abilita dopo lo Step 1).")
    print("4) Premi 'Scarica Excel'.")


HTML(value="<h3>GC-MS TOF HR Wizard (mzData.xml dentro ZIP) — Colab</h3><div style='padding:8px 10px; backgrou…

Accordion(children=(VBox(children=(HTML(value='<b>Step 1 — Estrazione compound (HR, seed-EIC)</b><br><small>Sl…

Output(layout=Layout(border='1px solid #ddd', max_height='420px', overflow_y='auto', padding='10px'))

Saving GC4_AU09.mzdata.zip to GC4_AU09.mzdata.zip


Estrazione per file:   0%|          | 0/3 [00:00<?, ?it/s]